# 08 modpack boot

Question: does Golem run the real SkyTekx 3 modpack, every mod of the live server unchanged, the
way the java launcher runs it, and what differs from Arclight, which the live server runs today.

The notebook takes the pack that `python3 -I tests/fetch.py --only modpack` fetches at the pinned
commit of the server repository: the 216 jars of `modrinth.index.json` that the server requires,
the 49 jars of `local-mods/`, `config/`, `defaultconfigs/`, `kubejs/` and the three datapacks of
`world/datapacks/`, with the jars patched by `tools/patch_mods.py` as on the live server. It scans
every class of the pack for Bukkit and Arclight, then copies the Forge server of `tests/fetch.py`
and the pack into a temporary directory and boots it from a fresh world with the settings of the
live server three times: under the java launcher as `run.sh` does, under `golem run` with the
bridge, and under the Arclight jar of the live server. A fourth boot runs golem on the world that
Arclight wrote, as the switch of the live server will. Each boot records `Done` and its time, the
mods that Forge loaded with their versions, the ERROR, FATAL and WARN lines, the crash reports, the
TPS and MSPT of a minute of idle, the answer to the server list and the stop. Golem must boot the
same mods as java with no error that java does not log. `okf/modpack.md` holds the numbers.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# Gradle, a path relative to the root or a name on PATH. tests/fetch.py installs it here.
gradle = "tests/.cache/gradle-8.14.3/bin/gradle"
# The JDK 17 whose libjvm golem loads and whose java runs the other boots and Gradle.
java_home = "tests/.cache/jdk-17.0.21+9"
# The Forge server that tests/fetch.py installs. The notebook boots copies of it alone.
forge_server = "tests/.cache/forge-server"
# The pack that tests/fetch.py --only modpack fetches. The notebook copies it and leaves it as it is.
modpack = "tests/.cache/modpack"
# The port of the copies of the server, which bind 127.0.0.1 alone.
server_port = 25677
# True passes --offline to Gradle, for a machine whose Gradle cache is warm.
gradle_offline = False
# The seconds a boot may take to reach Done, and a stop to end.
boot_timeout = 1800
# The seconds of idle after the server has started, and the seconds between two samples of the TPS.
idle_seconds = 60
sample_seconds = 10
# The seed of every world, so that the three boots generate the same world.
level_seed = "golem"
# True boots the pack a third time under the Arclight jar of the live server, with the libraries of
# its installer.json that tests/fetch.py fetched, so it downloads nothing at the start.
arclight = True
# True patches the jars of every copy with tools/patch_mods.py of the server, as the live server does.
jar_patches = True

# The pins: the commit of the server repository and what its pack holds.
server_commit = "3fa5ae681d12233fe81003fbfe5a2d3c529747fc"
index_jars = 216
local_jars = 49
patched_jars = 14
# The jars of the pack that Forge loads as a library, which forge mods does not list. Kotlin for Forge
# holds its mod, its language provider and its library as jars of JarJar.
library_jars = ["kotlinforforge-4.12.0-all.jar"]
# The mods that java loads from the jars of the pack and the jars nested in them, without
# minecraft and forge.
pack_mods = 289
datapacks = ["skytekx3_fixes", "skytekx3_skylands", "skytekx3_skylands_forge"]
forge_version = "47.4.18"
minecraft_version = "1.20.1"
# The options of the JVM of every boot, in user_jvm_args.txt for java and golem and on the command
# line of Arclight. The live server runs Aikar's flags with 11 GiB, which the notebook only reads.
jvm_options = ["-Xmx6G"]

## Setup

The cell finds the repository and the downloads, checks that the pack is the one of the pinned
commit, builds `golem` with cargo and reads what `golem source` names.

In [ ]:
import gzip
import hashlib
import io
import json
import os
import re
import shutil
import signal
import socket
import struct
import subprocess
import sys
import tempfile
import threading
import time
import tomllib
import weakref
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run
from harness.minecraft import PROTOCOL, Client, parse_varint, string, varint

P = paths(golem_root)


def located(value):
    # A value with a slash is a path relative to the root, a bare name is looked up on PATH.
    if "/" in value:
        return (P.root / value).resolve()
    found = shutil.which(value)
    assert found, "%s is not on PATH" % value
    return Path(found)


GRADLE = located(gradle)
JAVA_HOME = located(java_home)
FORGE = located(forge_server)
PACK = located(modpack)
for needed in (GRADLE, JAVA_HOME / "bin" / "java", FORGE):
    assert needed.exists(), "%s is missing, python3 -I tests/fetch.py installs it" % needed.name
assert (PACK / "modpack.json").is_file(), "the modpack is missing, python3 -I tests/fetch.py --only modpack fetches it"
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-08-"))
# The temporary directory goes when the kernel ends, after a failed cell too, unless the verdict
# removes it first. A finalizer needs an object to watch, and the kernel holds this one to its end.
SCRATCH_OWNER = type("ScratchOwner", (), {})()
CLEANUP = weakref.finalize(SCRATCH_OWNER, shutil.rmtree, str(SCRATCH), True)
OUT = P.out / "08_modpack_boot"
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
ENVIRONMENT = dict(os.environ, **QUIET, JAVA_HOME=str(JAVA_HOME))
WORKSPACE = tomllib.loads((P.root / "Cargo.toml").read_text())["workspace"]["package"]


def shown(text):
    # The text with the temporary directory and the root replaced, so no machine path is printed.
    return str(text).replace(str(SCRATCH), "<scratch>").replace(str(P.root), "<root>")


run(["cargo", "build", "--quiet", "-p", "golem"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
GOLEM = Path(metadata["target_directory"]) / "debug" / "golem"
assert GOLEM.is_file(), "cargo built no golem"
source = dict(line.split(" ", 1) for line in run([GOLEM, "source"], check=True).stdout.splitlines())
COMMIT = source["commit"]
print("python:", sys.version.split()[0])
print("golem source:", source)
assert source["version"] == WORKSPACE["version"] and source["repository"] == WORKSPACE["repository"], source

## The fetch of the pack

The pack comes from `tests/fetch.py --only modpack`, and the cells of this section hold the script
to what its docstring says without the network: they load it as a module and let it download from
a server of their own on 127.0.0.1, which answers as each case needs. A download that broke off
goes on with a range request, but only when the server answers with the bytes from that point on.
Wrong bytes before that point make the next try start over from the first byte, so a corrupt part
and a server that ignores the range both end with the right jar, and only a wrong file from the
first byte fails at once.

In [ ]:
import http.server
import importlib.util
import urllib.request

_spec = importlib.util.spec_from_file_location("golem_fetch", P.root / "tests" / "fetch.py")
FETCH = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(FETCH)
BODY = bytes(range(256)) * 80
BODY_SHA512 = hashlib.sha512(BODY).hexdigest()


class Mirror(http.server.BaseHTTPRequestHandler):
    # A server of one jar whose answer to a range request the case chooses, with two paths that
    # redirect and one whose answer has no end. It records every request with its Range header.
    mode, body, requests, sent, ended = "range", BODY, [], 0, threading.Event()

    def log_message(self, *arguments):
        pass

    def do_GET(self):
        Mirror.requests.append((self.path, self.headers.get("Range")))
        if self.path == "/to-http":
            return self.moved("http://127.0.0.1:%d/x.jar" % MIRROR.server_address[1])
        if self.path == "/to-other-host":
            return self.moved("https://example.invalid/x.jar")
        if self.path == "/endless":
            return self.endless()
        wanted, body = self.headers.get("Range"), Mirror.body
        start = int(wanted[len("bytes="):].rstrip("-")) if wanted else 0
        if wanted and Mirror.mode == "range":
            self.answer(206, body[start:], "bytes %d-%d/%d" % (start, len(body) - 1, len(body)))
        elif wanted and Mirror.mode == "whole 206":
            self.answer(206, body, "bytes 0-%d/%d" % (len(body) - 1, len(body)))
        elif wanted and Mirror.mode == "bare 206":
            self.answer(206, body)
        else:
            self.answer(200, body)

    def answer(self, status, body, content_range=None):
        self.send_response(status)
        if content_range:
            self.send_header("Content-Range", content_range)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def moved(self, location):
        self.send_response(302)
        self.send_header("Location", location)
        self.send_header("Content-Length", "0")
        self.end_headers()

    def endless(self):
        # Zeros until the client stops reading, cut at 64 MiB, counting what the socket took.
        self.send_response(200)
        self.end_headers()
        Mirror.sent = 0
        try:
            while Mirror.sent < 64 << 20:
                self.wfile.write(bytes(1 << 16))
                Mirror.sent += 1 << 16
        except OSError:
            pass
        finally:
            Mirror.ended.set()


MIRROR = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Mirror)
threading.Thread(target=MIRROR.serve_forever, daemon=True).start()
MIRROR_URL = "http://127.0.0.1:%d" % MIRROR.server_address[1]


def fetched(label, mode="range", part=None, body=BODY, path="/x.jar"):
    # Runs fetch_jar into a folder of its own, with `part` as the partial download, and answers
    # whether it downloaded, the bytes of the jar or the failure, the requests the server saw and
    # the files the folder holds afterwards.
    folder = SCRATCH / "fetch" / label
    folder.mkdir(parents=True)
    if part is not None:
        (folder / "x.jar.part").write_bytes(part)
    Mirror.mode, Mirror.body, Mirror.requests = mode, body, []
    entry = {"file": "x.jar", "url": MIRROR_URL + path, "sha512": BODY_SHA512, "size": len(BODY)}
    try:
        outcome = FETCH.fetch_jar(entry, folder)
        jar = (folder / "x.jar").read_bytes()
    except FETCH.Failure as failure:
        outcome, jar = None, str(failure)
    requests = list(Mirror.requests)
    print("%s: %s after %s" % (label, "the right jar" if jar == BODY else jar, requests))
    return outcome, jar, requests, sorted(path.name for path in folder.iterdir())


assert fetched("fresh")[:3] == (True, BODY, [("/x.jar", None)])
outcome, jar, requests, _ = fetched("good part", part=BODY[:5000])
assert (outcome, jar, requests) == (True, BODY, [("/x.jar", "bytes=5000-")]), requests
outcome, jar, requests, _ = fetched("corrupt part", part=bytes(5000))
assert (outcome, jar, requests) == (True, BODY, [("/x.jar", "bytes=5000-"), ("/x.jar", None)]), requests
for mode in ("whole 206", "bare 206"):
    outcome, jar, requests, _ = fetched(mode, mode=mode, part=BODY[:5000])
    assert (outcome, jar, requests) == (True, BODY, [("/x.jar", "bytes=5000-")]), (mode, requests)
outcome, message, requests, left = fetched("wrong file", body=bytes(len(BODY)))
assert outcome is None and "has SHA-512" in message and len(requests) == 1 and left == [], (message, requests, left)

The first download of an entry must name `cdn.modrinth.com` or `github.com`, the list of Golem,
and a download follows a redirect only over https to those hosts and to
`release-assets.githubusercontent.com`, where GitHub sends the download of a release. The
SHA-512 guards the bytes wherever they come from. A redirect to http or to any other host fails
at once, and the script reads no more of an answer than one byte past the size of the index, so
an answer without an end stops there.

In [ ]:
for path, needle in (("/to-http", "redirects to http on 127.0.0.1"), ("/to-other-host", "on example.invalid")):
    outcome, message, requests, left = fetched(path.strip("/"), path=path)
    assert outcome is None and needle in message and requests == [(path, None)] and left == [], (message, requests)
allowed = FETCH.Redirects().redirect_request(urllib.request.Request(MIRROR_URL + "/x.jar"), None, 302, "Found", {},
                                             "https://release-assets.githubusercontent.com/x.jar")
assert allowed.full_url == "https://release-assets.githubusercontent.com/x.jar", allowed.full_url
Mirror.ended.clear()
outcome, message, requests, left = fetched("endless", path="/endless")
assert Mirror.ended.wait(60), "the endless answer did not end"
print("the endless answer sent %d bytes before the client stopped reading" % Mirror.sent)
assert outcome is None and "is larger than the %d bytes" % len(BODY) in message and left == [], (message, left)
assert Mirror.sent < 64 << 20, "the script read the endless answer to its cut"

The index must hold what the format of Modrinth packs promises. Every entry names a jar in `mods/`
without a control character, an env that the format knows, a list of downloads, a SHA-512 of 128
lower case hex digits and a positive size, or the script refuses the index with a line that names
the entry. Any other failure of the fetch, an index that is no JSON or a file of the commit that
the data filter of `tarfile` refuses among them, ends it with one line `fetch: ...` and exit 1,
never with a traceback. Two names of jars that differ in case or in the form of Unicode alone
are one file on the disk of a Mac, so they count as the same jar, in the index and across the
index and `local-mods/`.

In [ ]:
import contextlib


def jar_bytes(files):
    # A jar, a zip of `files`, as bytes.
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, "w") as jar:
        for name, content in files.items():
            jar.writestr(name, content)
    return stream.getvalue()


FAKE_ARCLIGHT = jar_bytes({"META-INF/installer.json": json.dumps({"installer": {}, "libraries": {}})})
GIT = ["git", "-c", "user.name=Golem", "-c", "user.email=golem@example.invalid", "-c", "commit.gpgsign=false",
       "-c", "core.hooksPath=/dev/null"]


def fake_server(label, index, extra=None):
    # A git repository shaped like the server repository, with `index` as its modrinth.index.json
    # and a stand-in for every other file that the pack takes, in one commit. A value ("link",
    # target) is a symbolic link. Answers the folder and the commit.
    folder = SCRATCH / "fetch" / ("server-" + label)
    files = {"modrinth.index.json": index if isinstance(index, str) else json.dumps(index),
             "local-mods/local.jar": jar_bytes({"local.txt": "local"}), "config/a.toml": "a = 1\n",
             "defaultconfigs/b.toml": "b = 1\n", "kubejs/server_scripts/c.js": "// c\n",
             "world/datapacks/d/pack.mcmeta": "{}\n", "server.properties.example": "motd=x\n",
             "systemd/skytekx3.service": "[Service]\n", FETCH.ARCLIGHT_JAR: FAKE_ARCLIGHT, "arclight.conf": "",
             "bukkit.yml": "", "spigot.yml": "", "commands.yml": "", "tools/patch_mods.py": "print('patch_mods: done')\n",
             **(extra or {})}
    for name, content in files.items():
        path = folder / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if isinstance(content, tuple):
            path.symlink_to(content[1])
        elif isinstance(content, bytes):
            path.write_bytes(content)
        else:
            path.write_text(content)
    for arguments in (["init", "--quiet"], ["add", "--all"], ["commit", "--quiet", "-m", "pack"]):
        run(GIT + arguments, cwd=folder, check=True)
    return folder, run(["git", "rev-parse", "HEAD"], cwd=folder, check=True).stdout.strip()


@contextlib.contextmanager
def pinned(**values):
    # The module of the fetch with other pins for the time of the block.
    saved = {name: getattr(FETCH, name) for name in values}
    for name, value in values.items():
        setattr(FETCH, name, value)
    try:
        yield
    finally:
        for name, value in saved.items():
            setattr(FETCH, name, value)


def fetch_main(label, index, extra=None, cache=None, arclight=FAKE_ARCLIGHT, libraries=None):
    # Runs the main of the script with --only modpack on a fake server repository, as the command
    # line runs it, with the Arclight jar and the pins of its libraries given, and answers its exit
    # status, what it printed and the cache.
    repository, commit = fake_server(label, index, {FETCH.ARCLIGHT_JAR: arclight, **(extra or {})})
    cache = cache or SCRATCH / "fetch" / ("cache-" + label)
    out, err = io.StringIO(), io.StringIO()
    with pinned(SERVER_COMMIT=commit, ARCLIGHT_SHA256=hashlib.sha256(arclight).hexdigest(),
                ARCLIGHT_LIBRARIES=libraries or {}), contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
        status = FETCH.main(["--only", "modpack", "--cache", str(cache), "--server-repo", str(repository)])
    return status, out.getvalue(), err.getvalue(), cache


def index_of(*files):
    return {"formatVersion": 1, "game": "minecraft", "files": list(files),
            "dependencies": {"minecraft": minecraft_version, "forge": forge_version}}


GOOD = {"path": "mods/a.jar", "env": {"client": "required", "server": "required"}, "fileSize": 1,
        "downloads": ["https://cdn.modrinth.com/data/a/a.jar"], "hashes": {"sha512": "0" * 128}}
assert FETCH.server_files(index_of(GOOD)) == ([{"file": "a.jar", "url": GOOD["downloads"][0], "sha512": "0" * 128,
                                                 "size": 1}], [])
BROKEN = {
    "empty downloads": index_of(dict(GOOD, downloads=[])),
    "downloads no list": index_of(dict(GOOD, downloads=GOOD["downloads"][0])),
    "a download no text": index_of(dict(GOOD, downloads=[1])),
    "no fileSize": index_of({key: value for key, value in GOOD.items() if key != "fileSize"}),
    "fileSize 0": index_of(dict(GOOD, fileSize=0)),
    "fileSize text": index_of(dict(GOOD, fileSize="1")),
    "fileSize true": index_of(dict(GOOD, fileSize=True)),
    "no hashes": index_of({key: value for key, value in GOOD.items() if key != "hashes"}),
    "short sha512": index_of(dict(GOOD, hashes={"sha512": "0" * 127})),
    "upper case sha512": index_of(dict(GOOD, hashes={"sha512": "A" * 128})),
    "NUL in the path": index_of(dict(GOOD, path="mods/a\x00.jar")),
    "bidi override in the path": index_of(dict(GOOD, path="mods/a‮.jar")),
    "env no mapping": index_of(dict(GOOD, env="server")),
    "env unknown": index_of(dict(GOOD, env={"server": "maybe"})),
    "no path": index_of({key: value for key, value in GOOD.items() if key != "path"}),
    "entry no mapping": index_of("mods/a.jar"),
    "files no list": dict(index_of(), files={}),
    "dependencies no mapping": dict(index_of(GOOD), dependencies=None),
    "index no mapping": [GOOD],
}
for label, index in BROKEN.items():
    try:
        FETCH.server_files(index)
    except FETCH.Failure as failure:
        print("%s: %s" % (label, failure))
    else:
        raise AssertionError("an index with %s passed" % label)
CASES = [
    ("malformed entry", index_of(dict(GOOD, downloads=[])), None, "without a list of downloads"),
    ("no json", "{", None, "is no JSON"),
    ("link out of the tree", index_of(), {"local-mods/escape.jar": ("link", "../../outside.jar")}, "do not unpack"),
    ("twice in the index", index_of(GOOD, GOOD), None, "twice: a.jar and a.jar"),
    ("case against local-mods", index_of(dict(GOOD, path="mods/Local.jar")), None, "twice: Local.jar and local.jar"),
    ("form against local-mods", index_of(dict(GOOD, path="mods/café.jar")),
     {"local-mods/café.jar": jar_bytes({"c": "c"})}, "twice: café.jar and café.jar"),
]
for label, index, extra, needle in CASES:
    status, out, err, _ = fetch_main(label.replace(" ", "-"), index, extra)
    print("%s: exit %d, %s" % (label, status, shown(err.strip())))
    assert status == 1 and len(err.splitlines()) == 1 and err.startswith("fetch: ") and needle in err, (label, err)

The format of Modrinth packs makes `env` optional, and a launcher puts a file without it on both
sides, so the script fetches an entry without `env`, or with an `env` that leaves out the server,
for the server, and leaves out only an entry that the server does not require.

In [ ]:
without = {key: value for key, value in GOOD.items() if key != "env"}
for label, entry in (("no env", without), ("env of the client alone", dict(GOOD, env={"client": "required"}))):
    required, left_out = FETCH.server_files(index_of(entry))
    print("%s: fetched %s, left out %s" % (label, [jar["file"] for jar in required], left_out))
    assert [jar["file"] for jar in required] == ["a.jar"] and left_out == [], (label, required, left_out)
for side in ("optional", "unsupported"):
    assert FETCH.server_files(index_of(dict(GOOD, env={"client": "required", "server": side}))) == ([], ["a.jar"]), side

A pack of an earlier pin keeps no file that the new pin does not name, so an Arclight jar of an
older build cannot stay beside the new one, where a boot could pick it. The cell fetches a pack
from a fake server repository, plants the files of an earlier pin in it, fetches again and asserts
that they are gone, and that `modpack.json` names the Arclight jar with its SHA-256.

In [ ]:
status, out, err, cache = fetch_main("earlier-pin", index_of())
assert status == 0, err
pack = cache / FETCH.MODPACK_DIRECTORY
planted = ["arclight-forge-1.20.1-1.0.5-old.jar", "world/old-world/level.dat", "tools/old_patch.py", "systemd/old.service",
           "stray.txt", "mods/old-mod.jar"]
for name in planted:
    (pack / name).parent.mkdir(parents=True, exist_ok=True)
    (pack / name).write_text("old")
status, out, err, _ = fetch_main("this-pin", index_of(), cache=cache)
assert status == 0, err
left = [name for name in planted if (pack / name).exists()]
assert not left, "files of an earlier pin stay in the pack: %s" % left
assert (pack / "world" / "datapacks" / "d" / "pack.mcmeta").is_file() and (pack / "tools" / "patch_mods.py").is_file()
record = json.loads((pack / "modpack.json").read_text())
assert (record["arclight"]["file"], record["arclight"]["sha256"]) == (
    FETCH.ARCLIGHT_JAR, hashlib.sha256(FAKE_ARCLIGHT).hexdigest()), record["arclight"]
assert (pack / FETCH.ARCLIGHT_JAR).read_bytes() == FAKE_ARCLIGHT
print("the second fetch removed %s" % ", ".join(planted))

Arclight downloads the libraries that the `installer.json` of its jar names at its first start,
from mirrors of its own, and checks them by SHA-1 alone. The script fetches them into
`libraries/` of the pack instead, each by a pin of its URL, SHA-256 and size, and checks each
against the SHA-1 of `installer.json`, so a copy with them makes Arclight download nothing. A
library that `installer.json` names without a pin, or whose bytes miss its SHA-1, fails the fetch.

In [ ]:
LIBRARY = "org/example/lib/1/lib-1.jar"
PIN = {"org.example:lib:1": (MIRROR_URL + "/maven/", hashlib.sha256(BODY).hexdigest(), len(BODY))}


def arclight_naming(libraries):
    return jar_bytes({"META-INF/installer.json": json.dumps({"installer": {}, "libraries": libraries})})


status, out, err, cache = fetch_main("library", index_of(), libraries=PIN,
                                     arclight=arclight_naming({"org.example:lib:1": hashlib.sha1(BODY).hexdigest()}))
assert status == 0, err
pack = cache / FETCH.MODPACK_DIRECTORY
assert (pack / "libraries" / LIBRARY).read_bytes() == BODY
assert ("/maven/" + LIBRARY, None) in Mirror.requests, Mirror.requests
record = json.loads((pack / "modpack.json").read_text())["arclight"]["libraries"]
assert record == [{"coordinate": "org.example:lib:1", "file": LIBRARY, "sha256": PIN["org.example:lib:1"][1],
                   "sha1": hashlib.sha1(BODY).hexdigest()}], record
for label, named, needle in (("other SHA-1", {"org.example:lib:1": "0" * 40}, "would download it again"),
                             ("library without a pin", {"org.example:other:1": "0" * 40}, "name other libraries")):
    status, out, err, _ = fetch_main(label.replace(" ", "-"), index_of(), libraries=PIN, arclight=arclight_naming(named))
    print("%s: exit %d, %s" % (label, status, shown(err.strip())))
    assert status == 1 and len(err.splitlines()) == 1 and needle in err, (label, err)

Without `--server-repo` the script fetches the pinned commit of the server repository alone, into
a repository without a work tree, since it reads every file through `git show` and `git archive`,
and a branch keeps the commit. The cell lets it fetch a fake server repository by its URL.

In [ ]:
repository, commit = fake_server("by-url", index_of(), {FETCH.ARCLIGHT_JAR: FAKE_ARCLIGHT})
cache = SCRATCH / "fetch" / "cache-by-url"
with pinned(SERVER_REPOSITORY=repository.as_uri(), SERVER_COMMIT=commit, ARCLIGHT_LIBRARIES={},
            ARCLIGHT_SHA256=hashlib.sha256(FAKE_ARCLIGHT).hexdigest()):
    pack = FETCH.fetch_modpack(cache, None)
clone = cache / FETCH.SERVER_DIRECTORY
print("the clone holds %s" % sorted(path.name for path in clone.iterdir()))
assert sorted(path.name for path in clone.iterdir()) == [".git"], "the clone of the server has a work tree"
assert run(["git", "rev-parse", "refs/heads/pin"], cwd=clone, check=True).stdout.strip() == commit
assert json.loads((pack / "modpack.json").read_text())["commit"] == commit

## The bridge

The cell builds a copy of `bridge/` with Gradle, as `07_forge_boot` does, and keeps `golem-host.jar`
and the mod.

In [ ]:
TREE = SCRATCH / "tree"
shutil.copytree(P.root / "bridge", TREE / "bridge", ignore=shutil.ignore_patterns("build", ".gradle"))
shutil.copy2(P.root / "Cargo.toml", TREE / "Cargo.toml")
command = [GRADLE, "-p", TREE / "bridge", "--console=plain", "-Porg.gradle.java.installations.paths=%s" % JAVA_HOME,
           "build"] + (["--offline"] if gradle_offline else [])
started = time.monotonic()
build = subprocess.run([str(part) for part in command], capture_output=True, text=True, env=ENVIRONMENT, timeout=1700)
print("gradle build: exit %d after %.0f s" % (build.returncode, time.monotonic() - started))
assert build.returncode == 0, "the build failed:\n" + build.stdout[-6000:] + build.stderr[-6000:]
HOST_JAR = TREE / "bridge" / "host" / "build" / "libs" / "golem-host.jar"
MOD_JAR = TREE / "bridge" / "mod" / "build" / "libs" / ("golem-bridge-%s.jar" % WORKSPACE["version"])
assert HOST_JAR.is_file() and MOD_JAR.is_file(), (HOST_JAR, MOD_JAR)

## The pack

`modpack.json`, which `tests/fetch.py` writes last, must name the pinned commit, and the pack must
hold the 216 jars of the index and the 49 of `local-mods/`, each with the SHA-512 it names, the
three datapacks, the Arclight jar that `modpack.json` names with its SHA-256, the libraries of
its `installer.json` with theirs and `tools/patch_mods.py`, and nothing that the pin does not
name. The settings of the live server
come from its `server.properties.example`.

In [ ]:
def digest(path):
    state = hashlib.sha512()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            state.update(block)
    return state.hexdigest()


RECORD = json.loads((PACK / "modpack.json").read_text())
assert RECORD.get("schema") == "golem-modpack-2", "the pack is of the schema %s: refetch it" % RECORD.get("schema")
assert RECORD["commit"] == server_commit, "the pack is of %s, the pin is %s: refetch it" % (RECORD["commit"], server_commit)
assert RECORD["pack"]["forge"] == forge_version and RECORD["pack"]["minecraft"] == minecraft_version, RECORD["pack"]
PACK_JARS = RECORD["mods"]
sources = Counter(jar["source"] for jar in PACK_JARS)
assert sources == {"modrinth.index.json": index_jars, "local-mods": local_jars}, sources
on_disk = sorted(path.name for path in (PACK / "mods").iterdir())
assert on_disk == sorted(jar["file"] for jar in PACK_JARS), "mods/ of the pack holds other files than modpack.json names"
broken = [jar["file"] for jar in PACK_JARS if digest(PACK / "mods" / jar["file"]) != jar["sha512"]]
assert not broken, "jars that differ from their SHA-512: %s" % broken
assert sorted(path.name for path in (PACK / "world" / "datapacks").iterdir()) == datapacks
ARCLIGHT_JAR = RECORD["arclight"]["file"]
assert (ARCLIGHT_JAR, RECORD["arclight"]["sha256"]) == (FETCH.ARCLIGHT_JAR, FETCH.ARCLIGHT_SHA256), RECORD["arclight"]
found = hashlib.sha256((PACK / ARCLIGHT_JAR).read_bytes()).hexdigest()
assert found == RECORD["arclight"]["sha256"], "%s has SHA-256 %s, modpack.json names %s" % (
    ARCLIGHT_JAR, found, RECORD["arclight"]["sha256"])
assert (PACK / "tools" / "patch_mods.py").is_file()


def stray(folder, wanted, prefix=PurePosixPath()):
    # The paths below `folder` that are neither a path of `wanted` nor a folder on the way to one.
    for path in sorted(folder.iterdir()):
        relative = prefix / path.name
        if str(relative) in wanted:
            continue
        if path.is_dir() and any(PurePosixPath(name).is_relative_to(relative) for name in wanted):
            yield from stray(path, wanted, relative)
        else:
            yield str(relative)


NAMED = {"mods", "libraries", "modpack.json", *FETCH.MODPACK_TREES, *FETCH.MODPACK_FILES}
assert not list(stray(PACK, NAMED)), "the pack holds what the pin does not name: %s" % list(stray(PACK, NAMED))
LIBRARIES = RECORD["arclight"]["libraries"]
assert sorted(entry["coordinate"] for entry in LIBRARIES) == sorted(FETCH.ARCLIGHT_LIBRARIES), LIBRARIES
assert all(FETCH.ARCLIGHT_LIBRARIES[entry["coordinate"]][1] == entry["sha256"] == hashlib.sha256(
    (PACK / "libraries" / entry["file"]).read_bytes()).hexdigest() for entry in LIBRARIES), "a library misses its pin"
on_disk = sorted(str(path.relative_to(PACK / "libraries")) for path in (PACK / "libraries").rglob("*") if path.is_file())
assert on_disk == sorted(entry["file"] for entry in LIBRARIES), "libraries/ holds other files than modpack.json names"
LIVE_SETTINGS = {}
for line in (PACK / "server.properties.example").read_text().splitlines():
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        LIVE_SETTINGS[key] = value
print("pack %s %s of %s: %d jars of the index, %d of local-mods, %d left out as not for the server" % (
    RECORD["pack"]["name"], RECORD["pack"]["version"], server_commit[:12], sources["modrinth.index.json"],
    sources["local-mods"], len(RECORD["left_out"])))
print("live settings: level-type %s, view-distance %s, simulation-distance %s, max-tick-time %s" % (
    LIVE_SETTINGS["level-type"], LIVE_SETTINGS["view-distance"], LIVE_SETTINGS["simulation-distance"],
    LIVE_SETTINGS["max-tick-time"]))

## The options of the live JVM

The boots take a heap of 6 GiB alone, which fits the machines of the notebooks. The live server
starts its JVM with the options of its systemd unit, Aikar's flags with a heap of 11 GiB, and
`golem run` reads such options from `user_jvm_args.txt`, as `run.sh` does. The cell writes the
options of the unit into the `user_jvm_args.txt` of a launch and reads that launch with
`golem-devtool launch-spec`, the reader of `golem run`: every option of the unit must reach the JVM
unchanged and in its order.

In [ ]:
run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
unit = re.sub(r"\\\n", " ", (PACK / "systemd" / "skytekx3.service").read_text())
start = next(line for line in unit.splitlines() if line.startswith("ExecStart="))
tokens = start[len("ExecStart="):].split()
LIVE_OPTIONS = tokens[1:tokens.index("-jar")]
assert tokens[tokens.index("-jar"):] == ["-jar", "arclight.jar", "nogui"], tokens
assert "-Xmx11G" in LIVE_OPTIONS and "-XX:+UseG1GC" in LIVE_OPTIONS, LIVE_OPTIONS
LAUNCH = SCRATCH / "options"
LAUNCH.mkdir()
(LAUNCH / "libraries").symlink_to(FORGE / "libraries")
(LAUNCH / "user_jvm_args.txt").write_text("\n".join(LIVE_OPTIONS) + "\n")
spec = json.loads(run([DEVTOOL, "launch-spec", "--server", LAUNCH], check=True).stdout)
print("the unit names %d options: %s" % (len(LIVE_OPTIONS), " ".join(LIVE_OPTIONS)))
print("the launch holds %d options before %s" % (len(spec["options"]), spec["main_class"]))
assert spec["options"][:len(LIVE_OPTIONS)] == LIVE_OPTIONS, spec["options"][:len(LIVE_OPTIONS)]
assert spec["main_class"] == "cpw.mods.bootstraplauncher.BootstrapLauncher", spec["main_class"]

## No mod needs Bukkit

A mod that needs Bukkit or Arclight names their classes in its class files, or their names in its
`mods.toml`. The cell reads every class of every jar of the pack, jars nested in a jar included,
for the packages of Bukkit, CraftBukkit, Spigot, BungeeCord, Paper, Mohist and Arclight. No class
may name Bukkit or one of its forks. The only classes that name Arclight are the two mixins of
Supplementaries, which apply on Arclight and stay off without it. The cell also lists the classes
that install a handler of a signal, since `golem run` passes `-Xrs`, which leaves the signals of
the shutdown to the host: only the agent entry point of the debug probes of Kotlin, which runs only
when the probes are attached as a Java agent.

In [ ]:
NEEDLES = {
    "bukkit": (b"org/bukkit/", b"org.bukkit.", b"org/spigotmc/", b"org.spigotmc.", b"net/md_5/bungee",
               b"net.md_5.bungee", b"io/papermc/", b"io.papermc.", b"com/destroystokyo/paper", b"com/mohistmc",
               b"com.mohistmc"),
    "arclight": (b"io/izzel/arclight", b"io.izzel.arclight"),
    "signal": (b"sun/misc/Signal", b"jdk/internal/misc/Signal", b"sun.misc.Signal"),
}
TOML = re.compile(r"arclight|bukkit|spigot|mohist|papermc", re.IGNORECASE)


def scan(name, data, found, depth=0):
    with zipfile.ZipFile(io.BytesIO(data)) as jar:
        for info in jar.infolist():
            if info.filename.endswith(".jar") and depth < 3:
                scan(name + "!" + info.filename, jar.read(info), found, depth + 1)
            elif info.filename.endswith(".class"):
                body = jar.read(info)
                for kind, needles in NEEDLES.items():
                    if any(needle in body for needle in needles):
                        found[kind].add((name, info.filename))
            elif info.filename == "META-INF/mods.toml" and TOML.search(jar.read(info).decode("utf-8", "replace")):
                found["mods.toml"].add((name, info.filename))


started = time.monotonic()
FOUND = {"bukkit": set(), "arclight": set(), "signal": set(), "mods.toml": set()}
for jar in sorted((PACK / "mods").iterdir()):
    scan(jar.name, jar.read_bytes(), FOUND)
print("scanned %d jars in %.0f s" % (len(PACK_JARS), time.monotonic() - started))
for kind, hits in FOUND.items():
    print("%s: %d classes in %s" % (kind, len(hits), sorted({jar for jar, _ in hits}) or "no jar"))
    for jar, name in sorted(hits):
        print("   ", jar, name)
assert not FOUND["bukkit"], "classes that name Bukkit or a fork: %s" % sorted(FOUND["bukkit"])
assert not FOUND["mods.toml"], "mods.toml files that name Bukkit or Arclight: %s" % sorted(FOUND["mods.toml"])
assert {jar for jar, _ in FOUND["arclight"]} == {"supplementaries-1.20-3.1.44-forge.jar"}, FOUND["arclight"]
assert {jar for jar, _ in FOUND["signal"]} == {"kotlinforforge-4.12.0-all.jar"}, FOUND["signal"]
assert {name for _, name in FOUND["signal"]} == {"kotlinx/coroutines/debug/internal/AgentPremain.class"}, FOUND["signal"]

## A copy of the server

The cell copies the installed Forge server without its world, logs, mods and configuration, and
the pack into it: the mods, `config/`, `defaultconfigs/`, `kubejs/` and the datapacks in
`world/datapacks/`, so that Forge loads them into the new world as on the live server. It runs
`tools/patch_mods.py` of the server on the copy, which patches the jars of `mods/` beside it.
`server.properties` takes every setting of the live server, the level type with the Skylands
datapacks among them, but binds 127.0.0.1 on the port of the notebook, runs offline with a view
and simulation distance of 4 and RCON off, and seeds the world. Simple Voice Chat listens on a UDP
port of its own, 24454 in the pack, so the copy moves it to a port that is free, and a second run
or another program on 24454 cannot make one boot fail. `user_jvm_args.txt` holds the
options of the JVM, which `run.sh` and `golem run` read.

A boot starts the server, reads its console on a thread, and waits for `Done`. The server has
started when the console answers `forge tps`, after the listeners of `ServerStartedEvent`. Forge
writes the numbers of that answer in the default locale of the JVM, with a comma before the
decimals in some locales, so the parser reads a point and a comma alike, as the cell asserts. Then
the boot runs `forge mods`, asks for the server list and tries to log in a vanilla client, samples
`forge tps`, and under golem `golem status`, during the idle, reads the memory of the process and
stops the server: `stop` on the console under java and Arclight, `SIGTERM` under golem. A boot
that fails is killed, so no server outlives the cell.

In [ ]:
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
LOG_LINE = re.compile(r"^\[[^\]]*\] \[(?P<thread>[^\]]*)/(?P<level>[A-Z]+)\] \[(?P<logger>[^\]]*)\]: (?P<message>.*)$")
MOD_ROW = re.compile(r"^• (\S+) (.+?) : (\S+) \((.*)\) - (\d+)$")
FOUND_MOD = re.compile(r"Found valid mod file (.+?) with \{(.*?)\} mods - versions \{(.*?)\}$")
OVERALL = re.compile(r"Overall: Mean tick time: ([\d.,]+) ms\. Mean TPS: ([\d.,]+)")
GOLEM_STATUS = re.compile(r"Golem \S+: TPS ([\d.]+), MSPT ([\d.]+), ticks (\d+)")
DONE = re.compile(r"\]: Done \(([\d.]+)s\)! For help")
TOOK = re.compile(r"Dedicated server took ([\d.]+) seconds to load")
SKIP = {"world", "logs", "crash-reports", "mods", "config", "defaultconfigs"}
#: The settings of Simple Voice Chat, whose server listens on a UDP port of its own.
VOICE_CHAT = Path("config") / "voicechat" / "voicechat-server.properties"
#: A chunk that the IOWorker failed to write leaves a truncated entry in its region file, and every
#: load of it fails until the next save, which generates it again from scratch.
CHUNK_FAILURE = re.compile(r"Failed to (?:store|read) chunk")
VOICE_STARTED = re.compile(r"(?m)Voice chat server started at (?:\S+:|port )(\d+)\s*$")
ARCLIGHT_FILES = (ARCLIGHT_JAR, "arclight.conf", "bukkit.yml", "spigot.yml", "commands.yml")
UNIX_ARGS = "@libraries/net/minecraftforge/forge/%s-%s/unix_args.txt" % (minecraft_version, forge_version)
OVERRIDES = {"server-ip": "127.0.0.1", "server-port": str(server_port), "query.port": str(server_port),
             "online-mode": "false", "view-distance": "4", "simulation-distance": "4", "enable-rcon": "false",
             "level-seed": level_seed, "motd": "Golem modpack test"}
#: The loggers of the mod loader, whose WARN lines golem must share with java. The version check of
#: Forge asks the network, so its lines move with it.
LOADER = ("net.minecraftforge.fml.", "net.minecraftforge.jarjar.", "net.minecraftforge.registries.", "cpw.mods.",
          "mixin")
NOT_LOADER = ("net.minecraftforge.fml.VersionChecker",)


def tick_numbers(line):
    # The mean tick time and the TPS of the line Overall of forge tps. Forge writes them with a
    # DecimalFormat of the default locale of the JVM, which puts a comma before the decimals in
    # some locales and never groups the digits.
    return tuple(float(number.replace(",", ".")) for number in OVERALL.search(line).groups())


assert tick_numbers("Overall: Mean tick time: 1.760 ms. Mean TPS: 20.000") == (1.76, 20.0)
assert tick_numbers("Overall: Mean tick time: 1,760 ms. Mean TPS: 20,000") == (1.76, 20.0)


def free_port():
    # A UDP port on 127.0.0.1 that nothing holds now.
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


def prepare(kind):
    server = SCRATCH / kind
    shutil.copytree(FORGE, server, symlinks=True, ignore=lambda directory, names: [
        name for name in names if Path(directory) == FORGE and (name in SKIP or name.endswith(".log"))])
    for part in ("mods", "config", "defaultconfigs", "kubejs", "world/datapacks"):
        shutil.copytree(PACK / part, server / part)
    voice = server / VOICE_CHAT
    port = free_port()
    voice.write_text(re.sub(r"(?m)^port=\d+$", "port=%d" % port, voice.read_text()))
    assert re.findall(r"(?m)^port=(\d+)$", voice.read_text()) == [str(port)] and port != 24454, voice.read_text()
    patched = {}
    if jar_patches:
        (server / "tools").mkdir()
        shutil.copy2(PACK / "tools" / "patch_mods.py", server / "tools" / "patch_mods.py")
        result = run(["python3", "-I", server / "tools" / "patch_mods.py"], cwd=server, timeout=600)
        (OUT / ("%s-patch.log" % kind)).write_text(shown(result.stdout + result.stderr))
        assert result.returncode == 0 and "patch_mods: done" in result.stdout, shown(result.stdout + result.stderr)
        for jar in PACK_JARS:
            found = digest(server / "mods" / jar["file"])
            if found != jar["sha512"]:
                patched[jar["file"]] = (jar["source"], found)
    if kind == "golem":
        shutil.copy2(MOD_JAR, server / "mods" / MOD_JAR.name)
    if kind == "arclight":
        for name in ARCLIGHT_FILES:
            shutil.copy2(PACK / name, server / name)
        shutil.copytree(PACK / "libraries", server / "libraries", dirs_exist_ok=True)
    settings = {key: value for key, value in LIVE_SETTINGS.items() if key != "rcon.password"}
    settings.update(OVERRIDES)
    (server / "server.properties").write_text("".join("%s=%s\n" % item for item in settings.items()))
    (server / "user_jvm_args.txt").write_text("\n".join(jvm_options) + "\n")
    return server, patched


class Boot:
    # One run of a server, with its console read on a thread as it comes, each line with its time.

    def __init__(self, kind, server):
        self.kind, self.server = kind, server
        java = JAVA_HOME / "bin" / "java"
        if kind == "java":
            argv = [java, "@user_jvm_args.txt", UNIX_ARGS, "nogui"]
        elif kind == "golem":
            argv = [GOLEM, "run", "--server", server, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR, "--", "nogui"]
        else:
            argv = [java, *jvm_options, "-jar", ARCLIGHT_JAR, "nogui"]
        self.lines = []
        self.started = time.monotonic()
        self.process = subprocess.Popen([str(part) for part in argv], cwd=server, stdin=subprocess.PIPE,
                                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                        errors="replace", env=ENVIRONMENT)
        self.reader = threading.Thread(target=self.pump, daemon=True)
        self.reader.start()

    def pump(self):
        for line in self.process.stdout:
            self.lines.append((time.monotonic() - self.started, ANSI.sub("", line.rstrip("\n"))))

    def tail(self, count=40):
        return shown("\n".join(line for _, line in self.lines[-count:]))

    def expect(self, pattern, after=0, timeout=120):
        # Waits for a line from the index `after` on that matches `pattern`, and answers its index.
        pattern = re.compile(pattern) if isinstance(pattern, str) else pattern
        deadline = time.monotonic() + timeout
        while True:
            for index in range(after, len(self.lines)):
                if pattern.search(self.lines[index][1]):
                    return index
            if self.process.poll() is not None:
                raise AssertionError("%s ended with %s before %r:\n%s" % (
                    self.kind, self.process.returncode, pattern.pattern, self.tail()))
            if time.monotonic() > deadline:
                raise AssertionError("%s: no %r after %d s:\n%s" % (self.kind, pattern.pattern, timeout, self.tail()))
            time.sleep(0.2)

    def ask(self, command, pattern, timeout=120):
        # Runs a command on the console and answers the index of the first line of its answer.
        mark = len(self.lines)
        self.process.stdin.write(command + "\n")
        self.process.stdin.flush()
        return self.expect(pattern, mark, timeout)

    def memory(self):
        # The resident memory of the process now and at its peak, in MiB, on Linux.
        status = Path("/proc/%d/status" % self.process.pid)
        if not status.is_file():
            return None, None
        fields = dict(line.split(":", 1) for line in status.read_text().splitlines() if ":" in line)
        return tuple(int(fields[name].split()[0]) // 1024 for name in ("VmRSS", "VmHWM"))

    def kill(self):
        if self.process.poll() is None:
            self.process.kill()
            self.process.wait(60)
        self.reader.join(10)


def status(port):
    # The answer of the server to the server list, as JSON.
    with socket.create_connection(("127.0.0.1", port), timeout=60) as connection:
        stream = connection.makefile("rb")
        handshake = varint(PROTOCOL) + string("127.0.0.1") + struct.pack(">H", port) + varint(1)
        for body in (varint(0) + handshake, varint(0)):
            connection.sendall(varint(len(body)) + body)
        length, shift = 0, 0
        while True:
            byte = stream.read(1)[0]
            length |= (byte & 0x7F) << shift
            shift += 7
            if not byte & 0x80:
                break
        data = stream.read(length)
        _, offset = parse_varint(data)
        size, offset = parse_varint(data, offset)
        return json.loads(data[offset:offset + size].decode())


def forge_mods(answer):
    # The mods and the count of channels that Forge packs into the field d of forgeData: 15 bits
    # per character, then the mods with their channels, as ServerStatusPing of Forge 47 writes them.
    text = answer["forgeData"]["d"]
    size = ord(text[0]) | (ord(text[1]) << 15)
    data, buffer, bits = bytearray(), 0, 0
    for char in text[2:]:
        while bits >= 8:
            data.append(buffer & 0xFF)
            buffer, bits = buffer >> 8, bits - 8
        buffer |= (ord(char) & 0x7FFF) << bits
        bits += 15
    while len(data) < size:
        data.append(buffer & 0xFF)
        buffer, bits = buffer >> 8, bits - 8

    def utf(offset):
        length, offset = parse_varint(data, offset)
        return data[offset:offset + length].decode(), offset + length

    mods, channels, offset = {}, 0, 3
    for _ in range(int.from_bytes(data[1:3], "big")):
        flag, offset = parse_varint(data, offset)
        name, offset = utf(offset)
        version = None
        if not flag & 1:
            version, offset = utf(offset)
        for _ in range(flag >> 1):
            offset = utf(utf(offset)[1])[1] + 1
            channels += 1
        mods[name] = version
    return mods, channels


def nbt(data, offset, kind):
    # One tag of the named binary format of Minecraft, and the offset after it.
    fixed = {1: ">b", 2: ">h", 3: ">i", 4: ">q", 5: ">f", 6: ">d"}
    if kind in fixed:
        return struct.unpack_from(fixed[kind], data, offset)[0], offset + struct.calcsize(fixed[kind])
    if kind in (7, 11, 12):
        return None, offset + 4 + struct.unpack_from(">i", data, offset)[0] * {7: 1, 11: 4, 12: 8}[kind]
    if kind == 8:
        length = struct.unpack_from(">H", data, offset)[0]
        return data[offset + 2:offset + 2 + length].decode("utf-8", "replace"), offset + 2 + length
    if kind == 9:
        inner, count, offset = data[offset], struct.unpack_from(">i", data, offset + 1)[0], offset + 5
        items = []
        for _ in range(count):
            item, offset = nbt(data, offset, inner)
            items.append(item)
        return items, offset
    compound = {}
    while data[offset]:
        tag, length = data[offset], struct.unpack_from(">H", data, offset + 1)[0]
        name = data[offset + 3:offset + 3 + length].decode("utf-8", "replace")
        compound[name], offset = nbt(data, offset + 3 + length, tag)
    return compound, offset + 1


def level(server):
    data = gzip.decompress((server / "world" / "level.dat").read_bytes())
    root, _ = nbt(data, 3 + struct.unpack_from(">H", data, 1)[0], 10)
    values = root["Data"]
    return (values["SpawnX"], values["SpawnY"], values["SpawnZ"]), values["DataPacks"]["Enabled"]


def layout(server):
    # The folders of the world that hold the data of a dimension, which a loader must agree on to
    # open the world of another.
    world = server / "world"
    return sorted({str(path.parent.relative_to(world)) for path in world.rglob("*")
                   if path.is_dir() and path.name in ("data", "region") and "datapacks" not in path.parts})


def entries(server, text, level_name):
    # The lines of a level, each as its logger and its message with the paths, hashes and numbers
    # made alike, so that the lines of two boots compare.
    out = []
    for line in text.splitlines():
        match = LOG_LINE.match(line)
        if match and match["level"] == level_name:
            message = match["message"].replace(str(server), "<server>")
            message = re.sub(r"\d+", "N", re.sub(r"@[0-9a-f]{5,}|0x[0-9a-f]+", "@H", message))
            out.append((match["logger"], message))
    return out


def keep_logs(kind, server, session):
    # The log and the console of a boot in the output of the notebook, which CI keeps.
    if (server / "logs" / "latest.log").is_file():
        (OUT / ("%s-latest.log" % kind)).write_text(shown((server / "logs" / "latest.log").read_text(errors="replace")))
    (OUT / ("%s-console.log" % kind)).write_text(shown("\n".join(line for _, line in session.lines)) + "\n")


def parsed(server, console, rows):
    # What a boot that ended left in its console, its logs and its world.
    result = {}
    result["host"] = [line[len("[golem] "):] for line in console if line.startswith("[golem] ")]
    result["downloads"] = [line for line in console if "Downloading missing libraries" in line]
    result["took"] = next((float(TOOK.search(line)[1]) for line in console if TOOK.search(line)), None)
    latest = (server / "logs" / "latest.log").read_text(errors="replace")
    found = [FOUND_MOD.search(line) for line in (server / "logs" / "debug.log").read_text(errors="replace").splitlines()]
    found = [(match[1], match[2].split(","), match[3].split(",")) for match in found if match]
    result["mods"] = {}
    for provider, name, first, version, count in rows:
        hits = [(ids, versions) for file, ids, versions in found if file == name and ids[0] == first and versions[0] == version]
        ids, versions = hits[0] if hits else ([first], [version])
        assert len(ids) == int(count), (name, ids, count)
        result["mods"].update(zip(ids, versions))
    result["errors"] = entries(server, latest, "ERROR")
    result["fatal"] = entries(server, latest, "FATAL")
    result["warn"] = entries(server, latest, "WARN")
    result["loader_warn"] = {entry for entry in result["warn"]
                             if entry[0].startswith(LOADER) and not entry[0].startswith(NOT_LOADER)}
    result["error_lines"] = [shown(line) for line in latest.splitlines() if "/ERROR]" in line or "/FATAL]" in line]
    crashes = server / "crash-reports"
    result["crashes"] = sorted(path.name for path in crashes.iterdir()) if crashes.is_dir() else []
    result["spawn"], result["datapacks"] = level(server)
    result["layout"] = layout(server)
    stop = latest.rfind("Stopping server")
    result["flushed"] = stop >= 0 and "All dimensions are saved" in latest[stop:]
    result["chunk_failures"] = [shown(line) for line in latest.splitlines() if CHUNK_FAILURE.search(line)]
    result["saved"] = result["flushed"] and not result["chunk_failures"]
    result["voice"] = {"configured": int(re.search(r"(?m)^port=(\d+)$", (server / VOICE_CHAT).read_text())[1]),
                       "logged": [int(port) for port in VOICE_STARTED.findall(latest)]}
    return result


def boot(kind, launcher=None, server=None, keep=False):
    # Boots a fresh copy for `kind`, or the copy `server` that an earlier boot kept, under the
    # launcher, which is the kind unless named, and keeps the copy when asked.
    launcher = launcher or kind
    began = time.monotonic()
    result = {"kind": kind, "launcher": launcher, "patched": None}
    copy, session = server or SCRATCH / kind, None
    try:
        if server is None:
            server, result["patched"] = prepare(kind)
        result["prepared"] = time.monotonic() - began
        session = Boot(launcher, server)
        done = session.expect(DONE, timeout=boot_timeout)
        result["done"] = session.lines[done][0]
        result["done_step"] = float(DONE.search(session.lines[done][1])[1])
        started = session.ask("forge tps", OVERALL, timeout=boot_timeout)
        result["started"] = session.lines[started][0]
        listed = session.ask("forge mods", "Mod List:")
        session.ask("forge tps", OVERALL)
        result["rows"] = sorted({MOD_ROW.match(line.strip()).groups() for _, line in session.lines[listed:]
                                 if MOD_ROW.match(line.strip())})
        answer = status(server_port)
        result["ping"] = forge_mods(answer)
        result["ping_version"] = answer["version"]
        client = Client(server_port, "Tester", timeout=60)
        try:
            client.login()
            result["refusal"] = None
        except AssertionError as refused:
            result["refusal"] = re.sub(r"^.*?(\{.*\}).*$", r"\1", str(refused))
        finally:
            client.close()
        if launcher == "golem":
            session.ask("golem source", "Golem %s is free software" % re.escape(WORKSPACE["version"]))
            session.expect("Repository: " + re.escape(WORKSPACE["repository"]))
            if COMMIT != "unknown":
                session.expect("Source of this build: " + re.escape("%s/tree/%s" % (WORKSPACE["repository"], COMMIT)))
        result["tps"], result["golem_status"] = [], []
        # A sample every sample_seconds from the start of the idle, however long the answers take.
        idle = time.monotonic()
        for sample in range(1, idle_seconds // sample_seconds + 1):
            time.sleep(max(0.0, idle + sample * sample_seconds - time.monotonic()))
            result["tps"].append(tick_numbers(session.lines[session.ask("forge tps", OVERALL)][1]))
            if launcher == "golem":
                line = session.lines[session.ask("golem status", GOLEM_STATUS)][1]
                result["golem_status"].append(tuple(map(float, GOLEM_STATUS.search(line).groups())))
        result["idle"] = time.monotonic() - idle
        result["memory"] = session.memory()
        stopping = time.monotonic()
        mark = len(session.lines)
        if launcher == "golem":
            session.process.send_signal(signal.SIGTERM)
            session.expect(r"^\[golem\] SIGTERM: queued save-all flush and stop on the server thread", mark, 30)
        else:
            session.process.stdin.write("stop\n")
            session.process.stdin.flush()
        result["code"] = session.process.wait(boot_timeout)
        result["stop"] = time.monotonic() - stopping
        session.kill()
        keep_logs(kind, server, session)
        result.update(parsed(server, [line for _, line in session.lines], result["rows"]))
    except BaseException:
        # A boot that failed leaves its logs for the artifact of CI and no server behind it.
        if session is not None:
            session.kill()
            keep_logs(kind, copy, session)
        shutil.rmtree(copy, ignore_errors=True)
        raise
    if keep:
        result["server"] = server
    else:
        shutil.rmtree(server)
    tps = [sample[1] for sample in result["tps"]]
    print("%s: Done after %.0f s, started after %.0f s, %d mod files with %d mods, ERROR %d, FATAL %d, WARN %d, "
          "crash reports %d, TPS %.2f, exit %s after a stop of %.0f s" % (
              kind, result["done"], result["started"], len(result["rows"]), len(result["mods"]), len(result["errors"]),
              len(result["fatal"]), len(result["warn"]), len(result["crashes"]), sum(tps) / max(len(tps), 1),
              result["code"], result["stop"]))
    return result


BOOTS = {}
print("boot ready")

A boot that fails leaves its log, its console and the output of the jar patches in
`tests/notebooks/out/08_modpack_boot/` and no copy of the server behind it, and the temporary
directory of the run goes when the kernel ends, after a failed cell too. The cell boots a copy
under java with a deadline of one second for `Done`, which no server meets, and checks both.

In [ ]:
assert CLEANUP.alive and CLEANUP.atexit, "nothing removes the temporary directory when the kernel ends"
deadline, boot_timeout = boot_timeout, 1
try:
    boot("failing", launcher="java")
except AssertionError as failure:
    print("the boot failed as it should: %s" % shown(str(failure).splitlines()[0]))
else:
    raise AssertionError("a boot with a deadline of one second passed")
finally:
    boot_timeout = deadline
assert not (SCRATCH / "failing").exists(), "the failed boot left its copy of the server"
kept = sorted(path.name for path in OUT.iterdir() if path.name.startswith("failing-"))
print("kept: %s" % ", ".join(kept))
assert {"failing-console.log"} | ({"failing-patch.log"} if jar_patches else set()) <= set(kept), kept

## Under the java launcher

The first boot runs the copy as `run.sh` does, `java @user_jvm_args.txt @.../unix_args.txt nogui`,
and stops it with `stop` on the console, which must end java with 0. `tools/patch_mods.py` must
patch the 14 jars that the live server patches, 11 of the index and 3 of `local-mods/`.

In [ ]:
BOOTS["java"] = boot("java")
JAVA = BOOTS["java"]
print("patched: %s" % ", ".join(sorted(JAVA["patched"])))
if jar_patches:
    assert len(JAVA["patched"]) == patched_jars, sorted(JAVA["patched"])
    assert Counter(source for source, _ in JAVA["patched"].values()) == {"modrinth.index.json": 11, "local-mods": 3}
assert JAVA["code"] == 0, "java exited %s" % JAVA["code"]

## Under golem

The second boot runs the same copy under `golem run` with the host jar and the bridge mod. The
bridge must connect once, `golem source` must name the license, the repository and the tree of the
commit of the binary, `golem status` must count the ticks, and `SIGTERM` must stop the server
through the server thread with exit 0 after `DestroyJavaVM` answered 0. The patched jars must be
the same bytes as under java. Golem must log its banner and the four lines of a clean boot alone,
in their order: the mod connected the game, the bridge is connected, the stop on `SIGTERM` and the
end, so a panic of a plugin or a native, or a call over its budget, fails the boot.

In [ ]:
#: The lines that golem logs after its banner on a clean boot of the pack, in their order.
HOST_LINES = [
    "the mod connected the game, which the plugins may ask on the server thread",
    "the bridge is connected: Forge %s, Minecraft %s" % (forge_version, minecraft_version),
    "SIGTERM: queued save-all flush and stop on the server thread",
    "the server ended: main returned and DestroyJavaVM answered 0",
]


def clean_host(kind, host):
    # Asserts that golem logged its banner and the lines of a clean boot alone.
    print(shown("\n".join(host)))
    trouble = [line for line in host if "panicked" in line or "over the budget" in line]
    assert not trouble, "%s: golem logged a panic or a call over its budget: %s" % (kind, trouble)
    assert host and host[0].startswith("Golem %s at %s runs cpw.mods.bootstraplauncher.BootstrapLauncher of " % (
        WORKSPACE["version"], COMMIT)), host
    assert host[1:] == HOST_LINES, "%s: golem logged other lines than those of a clean boot: %s" % (kind, host[1:])


BOOTS["golem"] = boot("golem")
GOLEM_BOOT = BOOTS["golem"]
clean_host("golem", GOLEM_BOOT["host"])
assert GOLEM_BOOT["code"] == 0, "golem exited %s" % GOLEM_BOOT["code"]
assert GOLEM_BOOT["golem_status"] and all(ticks > 0 for _, _, ticks in GOLEM_BOOT["golem_status"]), GOLEM_BOOT["golem_status"]
assert GOLEM_BOOT["patched"] == JAVA["patched"], "the patched jars differ from those under java"

## Under Arclight

The third boot runs the Arclight jar of the live server, `java <options> -jar arclight.jar nogui`
as `ServerLinux.sh` does, with no Bukkit plugin, since the server repository tracks none. The copy
holds the libraries of Forge and those that the `installer.json` of Arclight names, which
`tests/fetch.py` fetched and pinned, so Arclight must download nothing: its console holds no line
`Downloading missing libraries`. `stop` on the console must end it with 0. `arclight = False`
leaves this boot and the next out.

In [ ]:
if arclight:
    BOOTS["arclight"] = boot("arclight", keep=True)
    ARCLIGHT = BOOTS["arclight"]
    banner = [row for row in ARCLIGHT["rows"] if row[0] == "arclight"]
    print("arclight:", banner)
    assert len(banner) == 1, ARCLIGHT["rows"]
    assert not ARCLIGHT["downloads"], "Arclight downloaded libraries: %s" % ARCLIGHT["downloads"]
    assert ARCLIGHT["code"] == 0, "arclight exited %s" % ARCLIGHT["code"]
    assert ARCLIGHT["patched"] == JAVA["patched"], "the patched jars differ from those under java"
else:
    print("arclight: left out")

## Golem on the world of Arclight

The live server moves from Arclight to golem with its world. The fourth boot runs `golem run` on
the copy that Arclight left, with the world that Arclight wrote and the bridge mod added. It must
reach `Done` with the mods of the golem boot, connect the bridge, log the lines of a clean boot of
golem alone, keep the spawn, log no ERROR line that java does not, and stop with exit 0 and every
dimension saved.

The WARN lines that golem logs on this world and not on its own must stand in the pinned list
below with their reasons, after two kinds of noise are made alike: the items of a list in brackets,
which follow the order of a hash set that changes from run to run, such as the invalid tags of
AlmostUnified, and the tag in which Redux found a holder twice, which follows the same order. The
two lines of the switch must be there. The fourth boot starts on the configuration that the third
left, so Forge corrects only the files that it corrects at every start, and `ticex-common.toml` of
TiCEX must be among them: TiCEX defines `avaritia.condensingDropProbability` with a default of the
type Float, which the parser of TOML reads back as a Double, so every start of the live server
logs both lines too.

In [ ]:
#: The WARN lines that golem logs on the world of Arclight and not on its own, after alike(), each
#: as its logger, a pattern of its message, whether the boot must log it, and the reason.
TAKEOVER_WARN = [
    ("net.minecraftforge.common.ForgeHooks/WP", r"The following mods have version differences that were not resolved:$",
     True, "Forge names the mod arclight, which the world was saved with and which is gone"),
    ("net.minecraft.server.MinecraftServer/", r"Missing data pack mod:arclight$", True,
     "the datapack of the mod arclight is gone with the mod"),
    ("net.minecraft.server.MinecraftServer/", r"Can't keep up! Is the server overloaded\?", False,
     "an overload of the shared machine, at random"),
    ("com.yanny.aci.manager.ManagedRegistry/", r"\[ali\] Missing function tooltips for ", False,
     "a loot function that Advanced Loot Info met without a tooltip, on the world of Arclight alone in the run of "
     "2026-11-06"),
]


def alike(entry):
    # A WARN line with the noise of the order of a hash set made alike: the items of a list in
    # brackets sorted, and the tag of a holder that Redux found twice dropped.
    logger, message = entry
    message = re.sub(r"\[([^\[\]]*, [^\[\]]*)\]", lambda found: "[%s]" % ", ".join(sorted(found[1].split(", "))), message)
    if logger == "net.zepalesque.redux.Redux/":
        message = re.sub(r" from tag \S+! ", " from a tag! ", message)
    return logger, message


def switched(logger, message):
    # The reason of the pinned line that a WARN line of the fourth boot is, or None.
    return next((reason for pinned, pattern, _, reason in TAKEOVER_WARN if logger == pinned and re.match(pattern, message)),
                None)


REDUX = ("net.zepalesque.redux.Redux/", "Found already existing holder minecraft:tundra from tag forge:is_sparse! Replacing "
         "value anyway! Old value was N, new is N")
assert alike(REDUX) == alike((REDUX[0], REDUX[1].replace("forge:is_sparse", "forge:is_magical"))), alike(REDUX)
assert alike(("AlmostUnified/", "ignored: [b:x, a:y]")) == ("AlmostUnified/", "ignored: [a:y, b:x]")
if arclight:
    shutil.copy2(MOD_JAR, ARCLIGHT["server"] / "mods" / MOD_JAR.name)
    BOOTS["takeover"] = boot("takeover", launcher="golem", server=ARCLIGHT["server"])
    TAKEOVER = BOOTS["takeover"]
    clean_host("takeover", TAKEOVER["host"])
    assert TAKEOVER["code"] == 0, "golem exited %s" % TAKEOVER["code"]
    assert TAKEOVER["mods"] == GOLEM_BOOT["mods"], "golem loads other mods on the world of Arclight"
    assert TAKEOVER["spawn"] == ARCLIGHT["spawn"], (TAKEOVER["spawn"], ARCLIGHT["spawn"])
    only = sorted(set(map(alike, TAKEOVER["warn"])) - set(map(alike, GOLEM_BOOT["warn"])))
    print("WARN lines of golem on the world of Arclight that golem on its own world lacks:")
    for logger, message in only:
        print("   ", logger, message[:200], "(%s)" % (switched(logger, message) or "not pinned"))
    unknown = [entry for entry in only if not switched(*entry)]
    assert not unknown, "golem logs WARN lines on the world of Arclight that the pinned list lacks: %s" % unknown
    for logger, pattern, required, reason in TAKEOVER_WARN:
        assert not required or any(switched(*entry) == reason for entry in only), "golem no longer logs %r" % pattern
    config = [message for logger, message in TAKEOVER["warn"] if logger == "net.minecraftforge.common.ForgeConfigSpec/CORE"]
    print("Forge corrects %d lines of configuration at the fourth start:" % len(config))
    for message in config:
        print("   ", message[:200])
    for pattern in (r"Configuration file \S*ticex-common\.toml is not correct\. Correcting$",
                    r"Incorrect key avaritia\.condensingDropProbability was corrected from N\.N to its default, N\.N\."):
        assert any(re.match(pattern, message) for message in config), "Forge no longer corrects %r" % pattern
else:
    print("takeover: left out")

## The comparison

The table holds what each boot recorded. `Done` counts from the start of the process to the line
`Done`, which closes the load of the worlds, and the start counts to the first answer of the
console after the listeners of `ServerStartedEvent`. The TPS and MSPT are those of `forge tps`
over the samples of the idle, and under golem also those of `golem status`. The memory is the
resident memory of the process at the end of the idle and at its peak.

In [ ]:
def mean(values):
    return sum(values) / len(values) if values else float("nan")


ORDER = [kind for kind in ("java", "golem", "arclight", "takeover") if kind in BOOTS]
ROWS = [
    ("Done after, s", lambda b: "%.0f" % b["done"]),
    ("Done line, s", lambda b: "%.1f" % b["done_step"]),
    ("Started after, s", lambda b: "%.0f" % b["started"]),
    ("ModernFix load, s", lambda b: "%.0f" % b["took"] if b["took"] else "none"),
    ("Mod files", lambda b: str(len(b["rows"]))),
    ("Mods with versions", lambda b: str(len(b["mods"]))),
    ("Mods in the server list", lambda b: str(len(b["ping"][0]))),
    ("Channels in the server list", lambda b: str(b["ping"][1])),
    ("Version in the server list", lambda b: b["ping_version"]["name"]),
    ("Patched jars", lambda b: str(len(b["patched"])) if b["patched"] is not None else "kept"),
    ("FATAL lines", lambda b: str(len(b["fatal"]))),
    ("ERROR lines", lambda b: str(len(b["errors"]))),
    ("WARN lines", lambda b: str(len(b["warn"]))),
    ("WARN lines of the loader", lambda b: str(len([e for e in b["warn"] if e[0].startswith(LOADER) and not e[0].startswith(NOT_LOADER)]))),
    ("Crash reports", lambda b: str(len(b["crashes"]))),
    ("Idle, s", lambda b: "%.0f" % b["idle"]),
    ("TPS mean of forge tps", lambda b: "%.2f" % mean([s[1] for s in b["tps"]])),
    ("TPS lowest of forge tps", lambda b: "%.2f" % min(s[1] for s in b["tps"])),
    ("MSPT mean of forge tps", lambda b: "%.2f" % mean([s[0] for s in b["tps"]])),
    ("MSPT highest of forge tps", lambda b: "%.2f" % max(s[0] for s in b["tps"])),
    ("TPS mean of golem status", lambda b: "%.2f" % mean([s[0] for s in b["golem_status"]]) if b["golem_status"] else "none"),
    ("MSPT mean of golem status", lambda b: "%.2f" % mean([s[1] for s in b["golem_status"]]) if b["golem_status"] else "none"),
    ("Memory now, MiB", lambda b: str(b["memory"][0])),
    ("Memory at the peak, MiB", lambda b: str(b["memory"][1])),
    ("Stop", lambda b: "SIGTERM" if b["launcher"] == "golem" else "stop"),
    ("Stop took, s", lambda b: "%.0f" % b["stop"]),
    ("Exit status", lambda b: str(b["code"])),
    ("Spawn", lambda b: "%d %d %d" % b["spawn"]),
    ("Datapacks enabled", lambda b: str(len(b["datapacks"]))),
]
width = max(len(label) for label, _ in ROWS)
print("%-*s  %s" % (width, "", "  ".join("%10s" % kind for kind in ORDER)))
for label, value in ROWS:
    print("%-*s  %s" % (width, label, "  ".join("%10s" % value(BOOTS[kind]) for kind in ORDER)))

## The same mods

Java must load every jar of the pack from `mods/`, but the jar of Kotlin for Forge, a library of
jars of JarJar that `forge mods` does not list, and no other file of `mods/`. Its other rows must be
Minecraft, Forge and the jars that JarJar takes out of the jars of the pack, and the mods of the
pack beside Minecraft and Forge must be the pinned count. Golem must load every mod that java loads,
each in the same version, from the same files, and nothing else but its bridge, the mod `golem`,
which the server list marks as needed on the server alone, so a client without it joins. Arclight
must load the same mods and its own. The server list must name the same mods under each, and refuse
a vanilla client with the same message of Forge. That refusal writes one ERROR line of the network
registry of Forge, with the channels that turn a vanilla client away, under each loader.

In [ ]:
JAVA_MODS = JAVA["mods"]
print("java: %d mods, %d of them in the server list" % (len(JAVA_MODS), len(JAVA["ping"][0])))
PACK_FILES = {jar["file"] for jar in PACK_JARS}
folder = {row[1] for row in JAVA["rows"] if row[0] == "mods_folder"}
assert set(library_jars) <= PACK_FILES and not folder & set(library_jars), library_jars
missing = sorted(PACK_FILES - folder - set(library_jars))
assert not missing, "jars of the pack that java does not load: %s" % missing
assert folder <= PACK_FILES, "java loads files of mods/ that the pack does not name: %s" % sorted(folder - PACK_FILES)
others = sorted(row for row in JAVA["rows"] if row[0] != "mods_folder")
assert sorted(row[2] for row in others if row[2] in ("minecraft", "forge")) == ["forge", "minecraft"], others
nested = [row for row in others if row[2] not in ("minecraft", "forge")]
assert all(row[0] == "JarInJar" for row in nested), [row for row in nested if row[0] != "JarInJar"]
OWN_MODS = sorted(key for key in JAVA_MODS if key not in ("minecraft", "forge"))
print("java loads %d jars of the pack from mods/, %s as a library, %d jars nested in them, and %d mods beside "
      "minecraft and forge" % (len(folder), ", ".join(library_jars), len(nested), len(OWN_MODS)))
assert len(OWN_MODS) == pack_mods, "java loads %d mods of the pack, the pin is %d" % (len(OWN_MODS), pack_mods)
extra = {key: value for key, value in GOLEM_BOOT["mods"].items() if key not in JAVA_MODS}
missing = sorted(set(JAVA_MODS) - set(GOLEM_BOOT["mods"]))
moved = sorted(key for key in JAVA_MODS if key in GOLEM_BOOT["mods"] and GOLEM_BOOT["mods"][key] != JAVA_MODS[key])
assert not missing and not moved, "golem lacks %s, and differs in the version of %s" % (missing, moved)
assert extra == {"golem": WORKSPACE["version"]}, extra
assert set(GOLEM_BOOT["rows"]) - set(JAVA["rows"]) == {("mods_folder", MOD_JAR.name, "golem", WORKSPACE["version"], "1")}
assert set(JAVA["rows"]) <= set(GOLEM_BOOT["rows"])
ping = {key: value for key, value in GOLEM_BOOT["ping"][0].items() if key != "golem"}
assert ping == JAVA["ping"][0] and "golem" in GOLEM_BOOT["ping"][0], "the server list under golem differs"
assert GOLEM_BOOT["ping"][0]["golem"] is None, "the server list asks clients for the bridge"
print("refusal of a vanilla client:", JAVA["refusal"])
assert JAVA["refusal"] and "require Forge to be installed on the client" in JAVA["refusal"], JAVA["refusal"]
for kind in ORDER:
    assert BOOTS[kind]["refusal"] == JAVA["refusal"], (kind, BOOTS[kind]["refusal"])
    assert BOOTS[kind]["ping_version"]["protocol"] == PROTOCOL, BOOTS[kind]["ping_version"]
    print("%s: the server list names %s" % (kind, BOOTS[kind]["ping_version"]))
for kind in ("java", "golem"):
    assert BOOTS[kind]["ping_version"]["name"] == minecraft_version, BOOTS[kind]["ping_version"]
if arclight:
    extra = {key: value for key, value in ARCLIGHT["mods"].items() if key not in JAVA_MODS}
    assert all(ARCLIGHT["mods"].get(key) == value for key, value in JAVA_MODS.items()), "Arclight differs in its mods"
    assert list(extra) == ["arclight"], extra
    assert {k: v for k, v in ARCLIGHT["ping"][0].items() if k != "arclight"} == JAVA["ping"][0]
    assert TAKEOVER["ping"] == GOLEM_BOOT["ping"], "the server list of golem on the world of Arclight differs"
for kind in ORDER:
    registry = [entry for entry in BOOTS[kind]["errors"] if entry[0] == "net.minecraftforge.network.NetworkRegistry/NETREGISTRY"]
    assert len(registry) == 1, "%s logs %d ERROR lines of the network registry for one refusal" % (kind, len(registry))
print("golem loads the %d mods of java and its bridge" % len(JAVA_MODS))

## Errors and warnings

No boot may write a crash report or a FATAL line. Every ERROR line under golem must appear under
java as well, or stand in the list of known lines below with its reason. A known line is one that
a server logs at random, so java may miss it in one run. A failed write or read of a chunk is no
known line, however random, since it loses the data of the chunk, and the check of the world fails
on it. The WARN lines of the loader, Forge
loading, JarJar, the registries, ModLauncher and Mixin, must be the same under golem as under java,
but for the version check of Forge, which asks the network.

In [ ]:
#: ERROR lines that a boot may log without the java boot logging them in the same run, each a
#: pattern of the message and the reason.
KNOWN = [
    (r"No key selector in MapLike", "Vanilla codec noise of new sculk sensors and shriekers while chunks generate"),
]


def explained(entry):
    return next((reason for pattern, reason in KNOWN if re.search(pattern, entry[1])), None)


for kind in ORDER:
    print("%s: ERROR and FATAL lines" % kind)
    for line in sorted(set(BOOTS[kind]["error_lines"]))[:40]:
        print("   ", re.sub(r"^\[[^\]]*\] ", "", line)[:220])
    assert not BOOTS[kind]["crashes"], (kind, BOOTS[kind]["crashes"])
    assert not BOOTS[kind]["fatal"], (kind, BOOTS[kind]["fatal"])
JAVA_ERRORS = set(JAVA["errors"])
for kind in ORDER[1:]:
    new = [entry for entry in set(BOOTS[kind]["errors"]) - JAVA_ERRORS if not explained(entry)]
    assert not new, "%s logs ERROR lines that java does not: %s" % (kind, new)
    known = [(entry, explained(entry)) for entry in set(BOOTS[kind]["errors"]) - JAVA_ERRORS]
    print("%s: ERROR lines beside those of java, all known: %s" % (kind, known or "none"))
only_java = sorted(JAVA["loader_warn"] - GOLEM_BOOT["loader_warn"])
only_golem = sorted(GOLEM_BOOT["loader_warn"] - JAVA["loader_warn"])
print("loader WARN kinds: java %d, golem %d" % (len(JAVA["loader_warn"]), len(GOLEM_BOOT["loader_warn"])))
assert not only_java and not only_golem, "loader WARN lines of java alone %s, of golem alone %s" % (only_java, only_golem)
loggers = Counter(logger for logger, _ in JAVA["warn"])
print("WARN lines by logger under java, the 12 largest:")
for logger, count in loggers.most_common(12):
    print("   %5d %s" % (count, logger))

## Idle, stop and world

The voice chat of every boot must have started on the free port of its copy. Over the idle the
server ticks at 20 TPS under java and golem, so the mean of the samples must be
19 or more for both. The boot takes one sample at every mark of `sample_seconds` from the start
of the idle, so it takes exactly `idle_seconds // sample_seconds` of them however long the console
takes to answer. The worlds come from the same seed, so their spawn and their datapacks must
be the same, with the three datapacks of the live server enabled. A mod with a `pack.mcmeta` is a
datapack of its own, so golem adds `mod:golem` and nothing else. Every loader must keep the data of
the dimensions in the same folders of the world, the layout of vanilla, and the stop must save every
dimension, which vanilla logs as `All dimensions are saved` once it has flushed them. Arclight logs
no such line, so its stop counts by its exit status alone. Vanilla logs that line even after the
IOWorker failed to write a chunk, which leaves a truncated entry in the region file, so no boot,
java and Arclight included, may log `Failed to store chunk` or `Failed to read chunk` anywhere,
and a boot counts as saved only with the line and without such a failure.

In [ ]:
for kind in ("java", "golem"):
    tps = mean([sample[1] for sample in BOOTS[kind]["tps"]])
    assert len(BOOTS[kind]["tps"]) == idle_seconds // sample_seconds and tps >= 19, (kind, BOOTS[kind]["tps"])
# The pack.mcmeta of a mod makes it a datapack of its own, so the bridge adds mod:golem.
OWN = {"java": set(), "golem": {"mod:golem"}, "arclight": {"mod:arclight"}, "takeover": {"mod:golem", "mod:arclight"}}
for kind in ORDER:
    assert BOOTS[kind]["spawn"] == JAVA["spawn"], (kind, BOOTS[kind]["spawn"], JAVA["spawn"])
    own = [name for name in BOOTS[kind]["datapacks"] if name not in JAVA["datapacks"]]
    print("%s: %d datapacks, beside those of java %s" % (kind, len(BOOTS[kind]["datapacks"]), own or "none"))
    assert set(own) <= OWN[kind], (kind, own)
    assert [name for name in BOOTS[kind]["datapacks"] if name not in own] == JAVA["datapacks"], kind
assert all("file/" + name in JAVA["datapacks"] for name in datapacks), JAVA["datapacks"]
assert "mod:golem" in BOOTS["golem"]["datapacks"]
for kind in ORDER:
    voice = BOOTS[kind]["voice"]
    assert voice["logged"] == [voice["configured"]] and voice["configured"] != 24454, (kind, voice)
    assert BOOTS[kind]["layout"] == JAVA["layout"], (kind, BOOTS[kind]["layout"], JAVA["layout"])
    failures = BOOTS[kind]["chunk_failures"]
    assert not failures, "%s failed to write or read a chunk, which loses its data: %s" % (kind, failures[:5])
    assert BOOTS[kind]["saved"] == (BOOTS[kind]["flushed"] and not failures), kind
    assert BOOTS[kind]["saved"] or kind == "arclight", "%s did not save every dimension at the stop" % kind
print("the dimensions hold their data in the same %d folders under every loader: %s" % (
    len(JAVA["layout"]), ", ".join(JAVA["layout"])))
print("spawn %s, the datapacks of the live server last: %s" % (JAVA["spawn"], JAVA["datapacks"][-3:]))

## What differs under Arclight

Arclight weaves its own mixins into the classes of Minecraft, and two of them meet mixins of the
pack. Its redirect `arclight$mutePotion` takes the call in `LivingEntity` that the Sundering of
Apothic Attributes redirects, so Sundering adds no damage on Arclight, and Apotheosis takes the
constant of the repair cost of the anvil that Arclight would raise. Neither line may appear under
golem, where both mixins of the pack apply as on plain Forge. The cell prints every other WARN of
the loader that only Arclight logs.

In [ ]:
CONFLICTS = [
    r"Skipping attributeslib\.mixins\.json:LivingEntityMixin->@Redirect::apoth_sunderingHasEffect.*"
    r"already redirected by mixins\.arclight\.core\.json",
    r"Skipping mixins\.arclight\.core\.json:world\.inventory\.RepairContainerMixin.*"
    r"already redirected by apotheosis\.mixins\.json:AnvilMenuMixin",
]
for pattern in CONFLICTS:
    assert not any(re.search(pattern, message) for _, message in GOLEM_BOOT["warn"]), pattern
if arclight:
    for pattern in CONFLICTS:
        assert any(re.search(pattern, message) for _, message in ARCLIGHT["warn"]), "Arclight no longer logs %r" % pattern
    print("loader WARN lines of Arclight alone:")
    for logger, message in sorted(ARCLIGHT["loader_warn"] - JAVA["loader_warn"]):
        print("   ", logger, message[:200])
    print("loader WARN lines that Arclight lacks:")
    for logger, message in sorted(JAVA["loader_warn"] - ARCLIGHT["loader_warn"]):
        print("   ", logger, message[:200])

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
CLEANUP()
print("verdict: golem run boots the %d mods of SkyTekx 3 at %s as java does, Done after %.0f s against %.0f s, "
      "with the same mods and versions, no FATAL line, no crash report, no ERROR line that java does not log, "
      "%.2f TPS in the idle and a stop with exit 0%s" % (
          len(OWN_MODS), server_commit[:12], GOLEM_BOOT["done"], JAVA["done"],
          mean([sample[1] for sample in GOLEM_BOOT["tps"]]),
          ", Arclight boots them too, Done after %.0f s, and golem opens the world that Arclight wrote" % ARCLIGHT["done"]
          if arclight else ""))